In [ ]:
####### this file has a new tokenization strategy to try to get my model to not be garbage
##### only real updates are: xml parsing does more to enforce global structure, viewbox handled
### specially in cleaning
## pre tokenizer implemented for whitepsace splitting

In [ ]:
#was having trouble with cairo svg working in ide managed jupyter environment
#explicitly setting env vars to resolve
import os
os.environ["DYLD_FALLBACK_LIBRARY_PATH"] = "/opt/homebrew/lib"
os.environ["DYLD_LIBRARY_PATH"] = "/opt/homebrew/lib"
os.environ["PKG_CONFIG_PATH"] = "/opt/homebrew/lib/pkgconfig"
import seaborn as sns
import re
from datasets import load_dataset
from lxml import etree
import random
import cairosvg
from PIL import Image
import io
import matplotlib.pyplot as plt
from datasets import concatenate_datasets
import numpy as np
from tokenizers import pre_tokenizers, trainers, models, Tokenizer, decoders
import global_config as cfg

PATH = "starvector/svg-icons-simple"
EMOJI_PATH = "starvector/svg-emoji-simple"
FONTS_PATH = "starvector/svg-fonts-simple"



In [ ]:
def hug_loadr(path):
    '''concatenates the default train test split fr HF to define custom later'''
    dset = load_dataset(path)
    dset = concatenate_datasets([
        dset["train"],
        dset["val"],
        dset["test"]
    ])
    dset = dset.filter(lambda x: x["Svg"] is not None)
    return dset

def is_xml_valid(example):
    'quick parse of xml'
    try:
        etree.fromstring(example["Svg"].encode("utf-8"))
        return True
    except Exception:
        return False


def normalize_svg(root, precision=1):
    GEOM = {"d", "points", "cx", "cy", "x", "y", "x1", "y1", "x2", "y2", "r", "rx", "ry", "width", "height"}

    def smart_round(val_str):
        #round and strip trailing zeros if present
        return re.sub(r"[-+]?\d*\.\d+|\d+", lambda m: f"{round(float(m.group()), precision):g}", val_str)

    for el in root.iter():
        #round attribute stuff
        for attr in (GEOM & set(el.attrib.keys())):
            el.attrib[attr] = smart_round(el.attrib[attr])

        # viewbox handle commas white sp
        if "viewBox" in el.attrib:
            vb = re.split(r'[,\s]+', el.attrib["viewBox"].strip())
            el.attrib["viewBox"] = " ".join([f"{round(float(x), precision):g}" for x in vb])

        # normalize whitesp
        if "d" in el.attrib:
            el.attrib["d"] = re.sub(r"\s+", " ", el.attrib["d"]).strip()

    return root

def xml_clean(example):
    if not example.get("Svg"): return {"Svg": None}
    try:
        parser = etree.XMLParser(remove_blank_text=True)
        root = etree.fromstring(example["Svg"].encode("utf-8"), parser=parser)
    except: return {"Svg": None}

    # remove metadata/namespaces
    etree.strip_elements(root, "metadata", "title", "desc", with_tail=False)
    for el in root.xpath("//*"):
        if '}' in str(el.tag): el.tag = el.tag.split('}', 1)[1]

    # normalize geometry and canonicalize attributes (sort alphabetically)
    root = normalize_svg(root)
    for el in root.iter():
        items = sorted(el.attrib.items())
        el.attrib.clear()
        el.attrib.update(items)

    # serialize, minify whitespace, and ensure xmlns exists for rendering
    svg = etree.tostring(root, encoding="unicode", method="xml", with_tail=False)
    svg = re.sub(r"\s+", " ", svg).replace("> <", "><").strip()

    if 'xmlns=' not in svg[:100]:
        svg = svg.replace('<svg ', '<svg xmlns="http://www.w3.org/2000/svg" ', 1)

    return {"Svg": svg}


def render_ok(example):
    '''render all of them as pngs to check their validity'''
    try:
        cairosvg.svg2png(bytestring=example["Svg"].encode("utf-8"))
        return True
    except Exception:
        return False







In [ ]:
def data_prep_pipeline(dset):
    #check xml validity, remove invalid
    dset = dset.filter(is_xml_valid)
    print(f"there are {len(dset)} entries with valid xml")
    #precision clean
    dset = dset.map(xml_clean)
    #drop nan
    dset = dset.filter(lambda x: x["Svg"] is not None)
    print(f"there are {len(dset)} entries after cleaning xml")

    #filter length of the SVG
    dset = dset.filter(lambda x: 50 < len(x["Svg"]) < cfg.SVG_THRESHOLD)
    print(f"there are {len(dset)} entries after removing short entries")
    #filter to only keep svgs that render
    dset = dset.filter(render_ok, num_proc=4)
    print(f"there are {len(dset)} entries in that redner")
    return dset



In [ ]:
def drop_filename(ds):
    return ds.remove_columns(["Filename"])
#load the three datasets (i came up wayyyy short on tokens and had to add in the other two)
#only reached about 15M trainable tokens using original ds, adding more
ds = hug_loadr(PATH)
print(f"there are {len(ds)} entries in {PATH}")
emoji = hug_loadr(EMOJI_PATH)
print(f"there are {len(emoji)} entries in {EMOJI_PATH}")
fonts = hug_loadr(FONTS_PATH)
print(f"there are {len(fonts)} entries in {FONTS_PATH}")

#save compute
ds = drop_filename(ds)
emoji= drop_filename(emoji)
fonts= drop_filename(fonts)

#subsample
emoji = emoji.shuffle(seed=cfg.RANDOM_SEED).select(range(min(50000, len(emoji))))
#this puts me at about 120M total all sets combined
fonts = fonts.shuffle(seed=cfg.RANDOM_SEED).select(range(min(1200000, len(fonts))))

#run the cleaning pipeline on all three sets
emoji = data_prep_pipeline(emoji)
fonts = data_prep_pipeline(fonts)
ds = data_prep_pipeline(ds)





In [ ]:
#add all three sets together
ds = concatenate_datasets([
    ds,
    emoji,
    fonts
])


In [ ]:
#use hugging face splitter
#using testsplit*2 to redivide again to get validation
split = ds.train_test_split(test_size=cfg.TEST_SPLIT*2, seed=cfg.RANDOM_SEED)
train_ds = split["train"]
temp_ds  = split["test"]

#subdivide test from above to get to get the validation/test sets
split2 = temp_ds.train_test_split(test_size=0.5, seed=cfg.RANDOM_SEED)
val_ds  = split2["train"]
test_ds = split2["test"]

In [ ]:
# Calculate lengths of all svgs
svg_lengths = [len(s) for s in test_ds['Svg']]

print(f"Min length: {min(svg_lengths)}")
print(f"Max length: {max(svg_lengths)}")

In [ ]:
#select random svgs (going to drop the svg column for mem mgmt so saving before tokenizing)
svg_examples = []

test_list = list(test_ds) # convert to list for easier random sampling
#get one random low/medium/high rez example
svg_examples.append(random.choice([x for x in test_list if len(x["Svg"]) < min(svg_lengths)+100]).copy())
svg_examples.append(random.choice([x for x in test_list if 500 <= len(x["Svg"]) <= max(svg_lengths)//2]).copy())
svg_examples.append(random.choice([x for x in test_list if len(x["Svg"]) > max(svg_lengths)-1000]).copy())


#syntax check print
print(svg_examples[:])

In [ ]:

# let tokenizer handle scheduler
os.environ["TOKENIZERS_PARALLELISM"] = "false"


TOKENIZER = Tokenizer(models.BPE())

TOKENIZER.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
TOKENIZER.decoder = decoders.ByteLevel()

special_tokens = ["[PAD]", "[BOS]", "[EOS]", "[UNK]"]

TRAINER = trainers.BpeTrainer(
    vocab_size=cfg.VOCAB_SIZE,
    min_frequency=2,
    special_tokens=special_tokens
)


#kept crashign on trainign much beyond this. thinking the length is long enough given that
#SVGs have highly repetitive entries... if natural language would need to expand
tokenizer_train_subset = train_ds.shuffle(seed=42).select(range(200000))

#iterator for train_from_iterator
def batch_iterator():
    for i in range(0, len(tokenizer_train_subset), 1000):
        yield tokenizer_train_subset[i : i + 1000]["Svg"]

#train
TOKENIZER.train_from_iterator(batch_iterator(), trainer=TRAINER)

#encode and return info on the run
def encode_batch(batch):
    encodings = TOKENIZER.encode_batch(batch["Svg"])
    return {
        "input_ids": [e.ids for e in encodings],
        "token_length": [len(e.ids) for e in encodings]
    }



In [ ]:
#encode/decode sanity check
sample_svg = '<svg xmlns="http://www.w3.org/2000/svg" height=>'

encoded = TOKENIZER.encode(sample_svg)
decoded = TOKENIZER.decode(encoded.ids)

print("--- RECONSTRUCTION CHECK ---")
print(f"Original: {sample_svg[:70]}...")
print(f"Decoded:  {decoded[:70]}...")

if sample_svg == decoded:
    print("\nyay it worked")
else:
    print("\ntry again")

In [ ]:

# get the number of available CPU cores div 2
CORES = cfg.CORES
#holding name, set tuples for plotting
plt_sets = []

#encoding loop
for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    print(f"Processing {name} set...")
    #map and remove string columns for memory
    processed = ds.map(encode_batch, batched=True,
                       batch_size=1000, num_proc=CORES, remove_columns=["Svg"]
    )
    #get rid of tokens that are too long
    processed = processed.filter(
        lambda x: 0 < x["token_length"] <= cfg.TOKEN_THRESHOLD,
        num_proc=CORES
    )
    plt_sets.append((name, processed))

# map back to og variables
train_ds = plt_sets[0][1]
val_ds   = plt_sets[1][1]
test_ds  = plt_sets[2][1]

In [ ]:
#get data info and print
train_token_count = sum(x["token_length"] for x in train_ds)
test_token_count  = sum(x["token_length"] for x in test_ds)
val_token_count   = sum(x["token_length"] for x in val_ds)

print(f"vocab size: {cfg.VOCAB_SIZE}")
print(f"train files: {len(train_ds)} | tokens: {train_token_count}")
print(f"test files: {len(test_ds)} | tokens: {test_token_count}")
print(f"val files: {len(val_ds)} | tokens: {val_token_count}")
print(f"total tokens: {train_token_count + test_token_count + val_token_count}")


In [ ]:
#####========= from here on out it's a plotting party

os.makedirs("visualizations", exist_ok=True)



#kde plots for token length distros
fig, ax = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
sns.set_style('whitegrid')
for i, (name, ds) in enumerate(plt_sets):
    sns.kdeplot(ds['token_length'], fill=True, ax=ax[i], alpha=0.4)
    ax[i].set_title(f"{name.upper()} Dist")
    ax[i].set_xlabel("Token length")


plt.suptitle("SVG Sequence Length Distributions")
plt.tight_layout()
plt.savefig("visualizations/token_set_len_distros.png", bbox_inches='tight', dpi=300)
plt.show()



In [ ]:

def show_svg(ex, title, tokenizer, save_path=None):
    '''renders an svg, counts token length'''
    png_bytes = cairosvg.svg2png(bytestring=ex["Svg"].encode("utf-8"))
    img = Image.open(io.BytesIO(png_bytes))

    white_bg = Image.new("RGBA", img.size, "WHITE")

    #get info to verify complexity
    token_count = len(tokenizer.encode(ex["Svg"]).ids)
    char_count = len(ex["Svg"])

    plt.figure(facecolor='white')
    plt.title(f"{title}\nTokens: {token_count} | Chars: {char_count}", color='black')
    plt.imshow(img)
    plt.axis("off")

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)

    plt.show()

complexities = ["LOW", "MED", "HIGH"]
#loop and plot
for i, c in zip(svg_examples, complexities):
    path = f"visualizations/{c}_complexity.png"
    show_svg(i, f"{c} complexity", TOKENIZER, save_path=path)

In [ ]:
#write all to bin format
train_bin = np.array([token for row in train_ds['input_ids'] for token in row], dtype=np.uint16)
train_bin.tofile("rob_train_set.bin")

test_bin = np.array([token for row in test_ds['input_ids'] for token in row], dtype=np.uint16)
test_bin.tofile("rob_test_set.bin")

val_bin = np.array([token for row in val_ds['input_ids'] for token in row], dtype=np.uint16)
val_bin.tofile("rob_val_set.bin")


In [ ]:
# save the tokenizer state
TOKENIZER.save("rob_tokenizer.json")